In [12]:
# 聚合    df.groupby("分组字段")["聚合字段"].聚合函数
# 聚合    df.groupby(["分组字段","分组字段"])[["聚合字段","聚合字段"]].聚合函数

import pandas as pd
df = pd.read_csv("../data/employees.csv")
# DataFrameGroupBy
dep_group = df.groupby("department_id")

# 查看分组信息   返回的是一个字典   key:部门id  value:属于当前组的所有数据索引列表
dep_group.groups

# 获取某一个分组数据
dep_group.get_group(50)

# 取出salary列的值   返回值是SeriesGroupBy   这个时候不会进行计算，只有调用聚合函数的时候，才会进行计算
dep_group["salary"]

# 计算
# dep_group["salary"].mean()
df.groupby("department_id")["salary"].mean()


for dept_id,group in df.groupby("department_id"):
    print(f"当前组为{dept_id}，组里的数据情况{group.shape}:")
    print(group.iloc[:,0:3])
    print("-------------------")


当前组为10.0，组里的数据情况(1, 10):
     employee_id first_name last_name
100          200   Jennifer    Whalen
-------------------
当前组为20.0，组里的数据情况(2, 10):
     employee_id first_name  last_name
101          201    Michael  Hartstein
102          202        Pat        Fay
-------------------
当前组为30.0，组里的数据情况(6, 10):
    employee_id first_name   last_name
14          114        Den    Raphaely
15          115  Alexander        Khoo
16          116     Shelli       Baida
17          117      Sigal      Tobias
18          118        Guy      Himuro
19          119      Karen  Colmenares
-------------------
当前组为40.0，组里的数据情况(1, 10):
     employee_id first_name last_name
103          203      Susan    Mavris
-------------------
当前组为50.0，组里的数据情况(45, 10):
    employee_id first_name    last_name
20          120    Matthew        Weiss
21          121       Adam        Fripp
22          122      Payam     Kaufling
23          123     Shanta      Vollman
24          124      Kevin      Mourgos
25          

In [19]:
# 按照多个字段进行分组
df1 = df.groupby(["department_id", "job_id"],as_index=False)[["salary", "commission_pct"]].mean()

df1
# df1.reset_index()



,department_id,job_id,salary,commission_pct
0,10.0,AD_ASST,4400.000000,NaN
1,20.0,MK_MAN,13000.000000,NaN
2,20.0,MK_REP,6000.000000,NaN
3,30.0,PU_CLERK,2780.000000,NaN
4,30.0,PU_MAN,11000.000000,NaN
5,40.0,HR_REP,6500.000000,NaN
6,50.0,SH_CLERK,3215.000000,NaN
7,50.0,ST_CLERK,2785.000000,NaN
8,50.0,ST_MAN,7280.000000,NaN
9,60.0,IT_PROG,5760.000000,NaN


In [34]:
# cut函数
df = pd.read_csv("../data/employees.csv")
pd.cut(df.iloc[9:16]["salary"],3)

pd.cut(df.iloc[9:16]["salary"],[0,10000,20000],labels=["low","high"])


9      low
10     low
11     low
12     low
13     low
14    high
15     low
Name: salary, dtype: category
Categories (2, object): ['low' < 'high']

In [42]:
# 按department_id分组，计算salary的最小值 ，中位数，最大值

# df.groupby("department_id")["salary"].min()
# df.groupby("department_id")["salary"].agg(["min","max","median"])

# 按department_id分组，统计job_id的种类数，commission_pct的平均值
df.groupby("department_id").agg({"job_id":"nunique","commission_pct":"mean"}).rename(
    columns={"job_id":"工种数","commission_pct":"平均值"}
)



,工种数,平均值
department_id,,
10.0,1,NaN
20.0,2,NaN
30.0,2,NaN
40.0,1,NaN
50.0,3,NaN
60.0,1,NaN
70.0,1,NaN
80.0,2,0.225
90.0,2,NaN


In [52]:
"""统计每个部门员工last_name的首字母"""
def f(x):
    """统计每个部门员工last_name的首字母"""
    result = set()
    for i in x:
        result.add(i[0])
    return result

df.groupby("department_id")["last_name"].agg(f)



department_id
10.0                                                   {W}
20.0                                                {F, H}
30.0                                    {K, C, H, T, B, R}
40.0                                                   {M}
50.0     {E, C, N, W, M, P, B, R, J, S, V, O, A, F, T, ...
60.0                                       {A, E, H, P, L}
70.0                                                   {B}
80.0     {E, C, M, P, B, R, J, S, V, O, H, A, Z, T, F, ...
90.0                                                {K, D}
100.0                                   {S, C, F, U, P, G}
110.0                                               {G, H}
Name: last_name, dtype: object

In [57]:
# 分组转换
df = pd.read_csv("../data/employees.csv")
# df.groupby("department_id")["salary"].mean()
df.groupby("department_id")["salary"].transform(lambda x: x - x.mean())





0      4666.666667
1     -2333.333333
2     -2333.333333
3      3240.000000
4       240.000000
          ...     
102   -3500.000000
103       0.000000
104       0.000000
105    1850.000000
106   -1850.000000
Name: salary, Length: 107, dtype: float64

In [72]:
# 2）通过transform()按分组使用平均值填充缺失值
# 读取员工数据
import numpy as np
df = pd.read_csv("../data/employees.csv")

na_index = pd.Series(df.index.tolist()).sample(30)  # 随机挑选30条数据
df.loc[na_index, "salary"] = pd.NA  # 将这30条数据的salary设置为缺失值
df.groupby("department_id")["salary"]
# df
# print(df.groupby("department_id")["salary"].agg(["size", "count"]))  # 查看每组数据总数与非空数据数
#
# def fill_missing(x):
#     # 使用平均值填充，如果平均值也为NaN，用0填充
#     if np.isnan(x.mean()):
#         return 0
#     return x.fillna(x.mean())
#
# df["salary"] = df.groupby("department_id")["salary"].transform(fill_missing)
# df.groupby("department_id")["salary"].transform(fill_missing)
# print(df.groupby("department_id")["salary"].agg(["size", "count"]))  # 查看每组数据总数与非空数据数


In [69]:
# 按department_id分组，过滤掉commission_pct包含空值的分组

df.groupby("department_id").filter(lambda x: x["commission_pct"].notnull().all())



,employee_id,first_name,last_name,email,phone_number,job_id,salary,commission_pct,manager_id,department_id
45,145,John,Russell,JRUSSEL,011.44.1344.429268,SA_MAN,14000.000000,0.40,100.0,80.0
46,146,Karen,Partners,KPARTNER,011.44.1344.467268,SA_MAN,13500.000000,0.30,100.0,80.0
47,147,Alberto,Errazuriz,AERRAZUR,011.44.1344.429278,SA_MAN,12000.000000,0.30,100.0,80.0
48,148,Gerald,Cambrault,GCAMBRAU,011.44.1344.619268,SA_MAN,11000.000000,0.30,100.0,80.0
49,149,Eleni,Zlotkey,EZLOTKEY,011.44.1344.429018,SA_MAN,10500.000000,0.20,100.0,80.0
50,150,Peter,Tucker,PTUCKER,011.44.1344.129268,SA_REP,10000.000000,0.30,145.0,80.0
51,151,David,Bernstein,DBERNSTE,011.44.1344.345268,SA_REP,9500.000000,0.25,145.0,80.0
52,152,Peter,Hall,PHALL,011.44.1344.478968,SA_REP,9000.000000,0.25,145.0,80.0
53,153,Christopher,Olsen,COLSEN,011.44.1344.498718,SA_REP,8000.000000,0.20,145.0,80.0
54,154,Nanette,Cambrault,NCAMBRAU,011.44.1344.987668,SA_REP,7500.000000,0.20,145.0,80.0
